# Applied Data Science Capstone - SpaceX Falcon 9 Landing Prediction

This notebook is structured for the final upload. It includes data loading, wrangling, EDA, SQL analysis, Folium mapping, Plotly visualization, and machine-learning classification.

In [ ]:
import pandas as pd, numpy as np, sqlite3
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
DATA_PATH = Path('spacex_launch_dataset.csv')
df = pd.read_csv(DATA_PATH)
df.head()

## Data wrangling
The target variable is `Class`, where 1 means successful first-stage landing and 0 means unsuccessful landing.

In [ ]:
df.info()
print(df.isna().sum())
df['PayloadMass'] = pd.to_numeric(df['PayloadMass'], errors='coerce')
df = df.dropna(subset=['PayloadMass','LaunchSite','Orbit','BoosterVersion','Class'])
df['Class'] = df['Class'].astype(int)
df.describe(include='all')

## Exploratory data analysis

In [ ]:
by_year = df.groupby('Year')['Class'].mean()
ax = by_year.plot(marker='o', title='Landing success rate by year')
ax.set_ylabel('Success rate')
ax.set_ylim(0,1)
plt.show()

In [ ]:
ax = df.groupby('LaunchSite')['Class'].mean().sort_values().plot(kind='barh', title='Success rate by launch site')
ax.set_xlabel('Success rate')
plt.show()

In [ ]:
ax = df.plot.scatter(x='PayloadMass', y='Class', alpha=.45, title='Payload mass vs landing outcome')
plt.show()

## SQL analysis

In [ ]:
con = sqlite3.connect(':memory:')
df.to_sql('launches', con, index=False, if_exists='replace')
pd.read_sql_query("""
SELECT LaunchSite, COUNT(*) AS launches, ROUND(AVG(Class), 3) AS success_rate, ROUND(AVG(PayloadMass), 1) AS avg_payload_kg
FROM launches
GROUP BY LaunchSite
ORDER BY success_rate DESC;
""", con)

In [ ]:
pd.read_sql_query("""
SELECT Orbit, COUNT(*) AS launches, ROUND(AVG(Class), 3) AS success_rate
FROM launches
GROUP BY Orbit
ORDER BY success_rate DESC;
""", con)

## Folium map and Plotly dashboard
Run these cells in a live notebook to display the interactive objects. HTML versions are included in the ZIP package.

In [ ]:
import folium
site_summary = df.groupby(['LaunchSite','Latitude','Longitude']).agg(launches=('Class','size'), success_rate=('Class','mean')).reset_index()
m = folium.Map(location=[df.Latitude.mean(), df.Longitude.mean()], zoom_start=4)
for _, r in site_summary.iterrows():
    folium.Marker([r.Latitude, r.Longitude], popup=f"{r.LaunchSite}: {r.launches} launches, {r.success_rate:.1%} success").add_to(m)
m

In [ ]:
import plotly.express as px
fig = px.scatter(df, x='PayloadMass', y='Class', color='LaunchSite', symbol='Orbit', hover_data=['Year','BoosterVersion'], title='Payload vs landing outcome')
fig.show()

## Machine learning

In [ ]:
features = ['FlightNumber','PayloadMass','Orbit','LaunchSite','BoosterVersion','Year']
X = df[features]
y = df['Class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.25, random_state=7, stratify=y)
preprocess = ColumnTransformer([
    ('num', StandardScaler(), ['FlightNumber','PayloadMass','Year']),
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['Orbit','LaunchSite','BoosterVersion'])
])
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'SVM': SVC(kernel='rbf', C=2),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=7),
    'KNN': KNeighborsClassifier(n_neighbors=7)
}
results=[]
for name, model in models.items():
    pipe = Pipeline([('prep', preprocess), ('model', model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    results.append((name, accuracy_score(y_test, preds), confusion_matrix(y_test, preds)))
pd.DataFrame([(n, round(a,3)) for n,a,c in results], columns=['Model','Test Accuracy']).sort_values('Test Accuracy', ascending=False)

## Conclusion
Success rates rise with time and newer booster versions. Launch site, orbit, and payload mass help explain the landing outcome. The classification model provides a useful estimate of first-stage landing success and can be improved with live API data, weather variables, booster reuse counts, and mission telemetry.